First, import the packages that you will need to use this notebook and mount your google drive if necessary.

In [ ]:
#Import Packages
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import norm

import time
import sys
import matplotlib.pyplot as plt
import math
import os

from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

Specify the path to your data.

In [ ]:
# Specify the path to your CSV file
file_path = r"Z:\MINGLE\Data\Melanoma\23_10_11_Melanoma_Marker_Cell_Neighborhood.csv"

# Read the CSV file
df = pd.read_csv(file_path)
cells = df#[df['donor'] == 'B004']
cells = cells#[cells['unique_region'] == 'B004_Ascending']

copy_cells = cells.copy()

# Print the shape of the DataFrame (rows, columns)
print(cells.shape)

# Optionally, to view the first few rows of the file
print(cells.head())

(5019159, 71)
   Unnamed: 0      CCR7     CD103     CD11b     CD11c     CD138      CD15  \
0           0 -0.362926 -0.729925 -0.150820 -0.363121 -0.193573 -0.376392   
1           1 -0.542319 -0.723530 -0.193650 -0.359325 -0.199382 -0.394698   
2           2 -0.444684 -0.497128 -0.329550 -0.292091 -0.201171 -0.202878   
3           3 -0.829088 -1.629941 -0.341772 -0.380618 -0.255592 -0.419650   
4           4  0.111211 -0.134976  3.659562  0.540194 -0.181382  5.446553   

      CD163      CD20     CD206  ...  Cell_Type_Common  Cell_Type_Sub  CD38  \
0 -0.277710 -0.096556 -0.326673  ...           Stromal  CD73+ Stromal   NaN   
1 -0.219857 -0.126534 -0.333797  ...           Stromal  CD73+ Stromal   NaN   
2 -0.309667 -0.135984 -0.347450  ...       Endothelial    Endothelial   NaN   
3 -0.115570 -0.118611  0.245775  ...           Stromal  CD73+ Stromal   NaN   
4 -0.212632 -0.125781 -0.246451  ...        Neutrophil     Neutrophil   NaN   

              filename  region      x      y    

In [ ]:
cells.columns.unique()

Index(['Unnamed: 0', 'CCR7', 'CD103', 'CD11b', 'CD11c', 'CD138', 'CD15',
       'CD163', 'CD20', 'CD206', 'CD21', 'CD25', 'CD3', 'CD31', 'CD4', 'CD45',
       'CD45RA', 'CD45RO', 'CD56', 'CD57', 'CD68', 'CD73', 'CD8', 'CLEC9A',
       'CXCL10', 'CXCL13', 'CXCL9', 'CXCR3', 'CXCR5', 'CollagenIV', 'EOMES',
       'EZH2', 'FAP', 'FoxP3', 'GATA3', 'GranzymeB', 'HLA-DR', 'ICOS', 'IL10',
       'IRF8', 'Ki67', 'LAG3', 'LDH-A', 'MHC-I', 'MMP9', 'Melan-A', 'PD-1',
       'PD-L1', 'PTEN', 'Podoplanin', 'SOX10/S100', 'TCF1/7', 'Tbet',
       'Vimentin', 'aSMA', 'bCatenin', 'pERK1/2', 'DAPI', 'cellid', 'donor',
       'CCL5', 'Cell_Type_Common', 'Cell_Type_Sub', 'CD38', 'filename',
       'region', 'x', 'y', 'Cell_Type', 'Overall_Cell_Type', 'Neighborhood'],
      dtype='object')

In [ ]:
cells['donor'].unique()

array(['21_06_23_Melanoma', '30_05_23_Melanoma', '13_06_23_Melanoma',
       '05_06_23_Melanoma', '14_06_23_Melanoma', '18_06_23_Melanoma',
       '23_06_23_Melanoma', '23_06_26_Melanoma', '23_06_30_Melanoma',
       '23_07_01_Melanoma', '23_07_03_Melanoma', '23_07_06_Melanoma'],
      dtype=object)

In [ ]:
cells['filename'].unique()

array(['21_06_23_reg001.tsv', '30_05_23_reg002.tsv',
       '30_05_23_reg001.tsv', '13_06_23_reg002.tsv',
       '05_06_23_reg003.tsv', '05_06_23_reg002.tsv',
       '05_06_23_reg001.tsv', '14_06_23_reg002.tsv',
       '14_06_23_reg001.tsv', '18_06_23_reg001.tsv',
       '23_06_23_reg002.tsv', '23_06_23_reg001.tsv',
       '23_06_26_reg003.tsv', '23_06_26_reg002.tsv',
       '23_06_30_reg001.tsv', '23_06_30_reg002.tsv',
       '23_07_01_reg002.tsv', '23_07_01_reg001.tsv',
       '23_07_03_reg002.tsv', '23_07_03_reg001.tsv',
       '23_07_06_reg002.tsv'], dtype=object)

In [ ]:
cells['Neighborhood'].unique()

array(['Stromal Enriched', 'Vasculature', 'Neutrophil Enriched',
       'Macrophage Enriched Immune', 'Vasculature & Immune',
       'Productive T cell & Tumor', 'Proliferating Tumor',
       'Inflamed Tumor', 'Tumor & Immune', 'Epithelial/Skin Appendages',
       'Resting Tumor', 'PDPN+ Stromal Enriched', 'Follicle',
       'Perivascular', 'Immune Infiltrate', 'DC Enriched Immune'],
      dtype=object)

Now, we will perform neighborhood analysis on the subset of data. We will use defined functions to split the image into windows and specify the number of k nearest neighbors that we would like to use for our analysis.

In [ ]:
#Function for getting neighborhood windows
def get_windows(job, n_neighbors):

    # Unpack the job tuple containing start_time, idx, tissue_name, and indices
    start_time, idx, tissue_name, indices = job

    # Record the current time to measure the duration of the job
    job_start = time.time()

    # Print a message indicating the start of the job
    print("Starting:", str(idx+1)+'/'+str(len(exps)), ': ' + exps[idx])

    # Get the subset of the dataset for the specific tissue
    tissue = tissue_group.get_group(tissue_name)

    # Extract the coordinates (X, Y) for the points to be fitted from the tissue subset
    to_fit = tissue.loc[indices][[X, Y]].values

    # Fit the NearestNeighbors model on the tissue's X, Y coordinates
    fit = NearestNeighbors(n_neighbors=n_neighbors).fit(tissue[[X, Y]].values)

    # Find the nearest neighbors for the points in 'to_fit'
    m = fit.kneighbors(to_fit)

    # Sort the neighbors
    # 'args' are the indices that would sort the distances
    args = m[0].argsort(axis=1)

    # 'add' is used to adjust indices for flattened array
    add = np.arange(m[1].shape[0]) * m[1].shape[1]

    # Calculate sorted indices for neighbors
    sorted_indices = m[1].flatten()[args + add[:, None]]

    # Retrieve the neighbor indices from the tissue dataset
    neighbors = tissue.index.values[sorted_indices]

    # Record the end time of the job
    end_time = time.time()

    # Print a message indicating the end of the job and the duration
    print("Finishing:", str(idx+1)+"/"+str(len(exps)), ": "+ exps[idx], end_time - job_start, end_time - start_time)

    # Return the neighbor indices as an array of integers
    return neighbors.astype(np.int32)

The next few blocks of code will help the code find cellhier to run neighborhood analysis.

In [ ]:
cellhier_path = r'Z:\MINGLE\Code\cellhier'
sys.path.append(r'Z:\MINGLE\Code\cellhier')
from general import *
from plot_john import *

Reset the dataframe index to ensure that neighborhood analysis can continue properly.

In [ ]:
cells.reset_index(inplace=True, drop=True)
print(cells)

         Unnamed: 0      CCR7     CD103     CD11b     CD11c     CD138  \
0                 0 -0.362926 -0.729925 -0.150820 -0.363121 -0.193573   
1                 1 -0.542319 -0.723530 -0.193650 -0.359325 -0.199382   
2                 2 -0.444684 -0.497128 -0.329550 -0.292091 -0.201171   
3                 3 -0.829088 -1.629941 -0.341772 -0.380618 -0.255592   
4                 4  0.111211 -0.134976  3.659562  0.540194 -0.181382   
...             ...       ...       ...       ...       ...       ...   
5019154     5019154  0.098215 -0.083622 -0.217149 -0.437058 -0.297181   
5019155     5019155 -0.240248 -0.113821 -0.464631 -0.433815 -0.275257   
5019156     5019156 -0.423292 -0.166772 -0.623979 -0.435652 -0.300200   
5019157     5019157  1.402914  0.407268  1.345068 -0.353550 -0.301476   
5019158     5019158 -0.319492  0.058658  0.715401 -0.436274  0.087566   

             CD15     CD163      CD20     CD206  ...  Cell_Type_Common  \
0       -0.376392 -0.277710 -0.096556 -0.326673  

In [ ]:
# Define column names that will be used for neighborhood analysis
X = 'x'                  # Variable for the X coordinate
Y = 'y'                  # Variable for the Y coordinate
reg = 'filename'         # Variable for the filename or region identifier associated with coordinates
cluster_col = 'Cell_Type'  # Variable for cell type/subtype classification

# List of columns to keep for analysis
keep_cols = [X, Y, reg, cluster_col]

In [ ]:
# Concatenate the original 'cells' DataFrame with dummy variables created from 'cluster_col'
# pd.get_dummies() converts categorical variable(s) into dummy/indicator variables
cells = pd.concat([cells, pd.get_dummies(cells[cluster_col])], axis=1)

# Get unique values from the 'cluster_col' column to use for summarization
sum_cols = cells[cluster_col].unique()

# Retrieve the values for these unique categories as a NumPy array
# This array can be used for further analysis or operations later for calculating the neighborhoods
values = cells[sum_cols].values

In [ ]:
#We can choose a range of nearest neighbors to calculate the neighborhoods
ks = [5,10,20] # k=5 means it collects 5 nearest neighbors for each center cell
n_neighbors = max(ks) #sets n_neighbors to max of the list that is set

In [ ]:
# Group the cell data by region
# 'cells' is a DataFrame containing cell data
# 'tissue_group' will be a GroupBy object with cells grouped by the 'reg' column (representing regions)
tissue_group = cells[[X, Y, reg]].groupby(reg)

# Get a list of unique regions (filenames)
# 'exps' will contain all unique region names found in the 'reg' column of the 'cells' DataFrame
exps = list(cells[reg].unique())

# Prepare chunks of data for processing
# 'tissue_chunks' is a list of tuples, each tuple representing a job for processing
# Each tuple contains the current time, index of the region in 'exps', the region name, and a subset of indices
# 'np.array_split(indices, 1)' splits the indices for each group into chunks (1 chunk in this case)
# This loop goes through each group in 'tissue_group', and for each group, it creates a job tuple
tissue_chunks = [(time.time(), exps.index(t), t, a) for t, indices in tissue_group.groups.items() for a in np.array_split(indices, 1)]

# Process each job to get the windows (neighbors of the cells)
# 'tissues' is a list of results from the 'get_windows' function
# The 'get_windows' function is applied to each job in 'tissue_chunks'
# 'n_neighbors' is a parameter for the 'get_windows' function, defining the number of neighbors to consider
tissues = [get_windows(job, n_neighbors) for job in tissue_chunks]

Starting: 7/21 : 05_06_23_reg001.tsv
Finishing: 7/21 : 05_06_23_reg001.tsv 0.2607243061065674 0.2607243061065674
Starting: 6/21 : 05_06_23_reg002.tsv
Finishing: 6/21 : 05_06_23_reg002.tsv 0.8278651237487793 1.0948946475982666
Starting: 5/21 : 05_06_23_reg003.tsv
Finishing: 5/21 : 05_06_23_reg003.tsv 0.42787623405456543 1.5322506427764893
Starting: 4/21 : 13_06_23_reg002.tsv
Finishing: 4/21 : 13_06_23_reg002.tsv 0.08458948135375977 1.6231913566589355
Starting: 9/21 : 14_06_23_reg001.tsv
Finishing: 9/21 : 14_06_23_reg001.tsv 1.109095573425293 2.7322869300842285
Starting: 8/21 : 14_06_23_reg002.tsv
Finishing: 8/21 : 14_06_23_reg002.tsv 0.8345208168029785 3.584639072418213
Starting: 10/21 : 18_06_23_reg001.tsv
Finishing: 10/21 : 18_06_23_reg001.tsv 4.483627796173096 8.078686475753784
Starting: 1/21 : 21_06_23_reg001.tsv
Finishing: 1/21 : 21_06_23_reg001.tsv 6.575530052185059 14.70626425743103
Starting: 12/21 : 23_06_23_reg001.tsv
Finishing: 12/21 : 23_06_23_reg001.tsv 0.11028289794921875 1

In [ ]:
# Initialize a dictionary to store the output
out_dict = {}

# Loop over a list of values 'ks' (different numbers of neighbors to consider)
for k in ks:
    # Iterate over each tissue's neighbors and the corresponding job information
    for neighbors, job in zip(tissues, tissue_chunks):

        # Create an array of indices for the current chunk of data
        chunk = np.arange(len(neighbors))  # equivalent to 0, 1, 2, ..., len(neighbors)-1

        # Extract the tissue name and indices from the job tuple
        tissue_name = job[2]  # Region/filename from the job tuple
        indices = job[3]      # Indices from the job tuple

        # Compute the 'window' - a summary measure for the neighborhood of each cell up to the k-th neighbor
        # Reshape and sum values to get a compact representation of neighborhood information
        window = values[neighbors[chunk, :k].flatten()].reshape(len(chunk), k, len(sum_cols)).sum(axis=1)

        # Store the computed window and indices in the output dictionary
        # Keyed by a tuple of (tissue_name, k)
        out_dict[(tissue_name, k)] = (window.astype(np.float16), indices)

# Initialize a dictionary to store the final windows data
windows = {}

# Iterate over each value of k again to process the stored information
for k in ks:

    # Concatenate data for each experiment ('exp') into a DataFrame
    # This DataFrame contains the window data for each cell, indexed by cell indices, for the current value of k
    window = pd.concat([pd.DataFrame(out_dict[(exp, k)][0], index=out_dict[(exp, k)][1].astype(int), columns=sum_cols) for exp in exps], axis=0)

    # Ensure the window data is in the same order as the original cells DataFrame
    window = window.loc[cells.index.values]

    # Concatenate the window data with the original columns specified in 'keep_cols'
    window = pd.concat([cells[keep_cols], window], axis=1)

    # Store the concatenated DataFrame in the 'windows' dictionary, keyed by the current value of k
    windows[k] = window


In [ ]:
#Choose k value to analyze and pull out from dictionary of stored results of vectors
k = 10
windows2 = windows[k]
#Add cell type column to output windows dataframe
windows2[cluster_col] = cells[cluster_col]

In [ ]:
windows2

c:\Users\k1van\anaconda3\envs\gpu_env\Lib\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
c:\Users\k1van\anaconda3\envs\gpu_env\Lib\site-packages\pandas\io\formats\format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,x,y,filename,Cell_Type,CD73+ Stromal,Endothelial,Neutrophil,DC,FAP+ Fibroblast,CD163+ CD206+ Macrophage,...,MHCI+ Tumor,MHCI+ PDL1+ Tumor,EZH2+ Tumor,EZH2+ Ki67+ Tumor,PDL1+ CD56+ Tumor,Bcatenin+ Tumor,PDPN+ Stromal,B,FDCs,LDHA+ SOX10/S100+ Tumor
0,3981,47146,21_06_23_reg001.tsv,CD73+ Stromal,8.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5286,39373,21_06_23_reg001.tsv,CD73+ Stromal,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,22899,42306,21_06_23_reg001.tsv,Endothelial,0.0,3.0,0.0,2.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5747,45479,21_06_23_reg001.tsv,CD73+ Stromal,9.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,17460,49393,21_06_23_reg001.tsv,Neutrophil,0.0,0.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5019154,4283,6987,23_07_06_reg002.tsv,Stromal,3.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5019155,17172,6095,23_07_06_reg002.tsv,MelanA+ Tumor,2.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5019156,16878,4870,23_07_06_reg002.tsv,Stromal,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5019157,13326,10841,23_07_06_reg002.tsv,Tumor,1.0,1.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


At the end of neighborhood analysis we have a dataframe containing each cell, its x and y coordinates, unique region, cell type, and counts of the cell types of their nearest neighbors. We will now use this dataframe to use mixture models to assign probabilities that a cell belongs to already annotated neighborhoods from the dataset.

Calculate Means and Standard Deviations for Cell Type Vector Features

In [ ]:
windows2.columns

Index(['x', 'y', 'filename', 'Cell_Type', 'CD73+ Stromal', 'Endothelial',
       'Neutrophil', 'DC', 'FAP+ Fibroblast', 'CD163+ CD206+ Macrophage',
       'Stromal', 'CD56+ Tumor', 'Smooth Muscle', 'Epithelial', 'CD8+ T',
       'CD68+ Macrophage', 'CD4+ T', 'CD4+ Treg', 'GZMB+ CD8+ T',
       'TCF1/7+ CD8+ T', 'GZMB+ PD1+ CD8+ T', 'PD1+ CD8+ T',
       'PD1+ TCF1/7+ CD8+ T', 'GZMB+ TCF1/7+ CD8+ T',
       'GZMB+ TCF1/7+ PD1+ CD8+ T', 'NK', 'Nerve', 'Tumor', 'MelanA+ Tumor',
       'SOX10/S100+ Tumor', 'PTEN+ Tumor', 'FAP+ Tumor', 'LDHA+ Tumor',
       'MHCI+ Tumor', 'MHCI+ PDL1+ Tumor', 'EZH2+ Tumor', 'EZH2+ Ki67+ Tumor',
       'PDL1+ CD56+ Tumor', 'Bcatenin+ Tumor', 'PDPN+ Stromal', 'B', 'FDCs',
       'LDHA+ SOX10/S100+ Tumor'],
      dtype='object')

In [ ]:
filtered_cells = copy_cells # make a copy of the original annotated dataset to ensure that we do not write over top of it
filtered_cells.reset_index(inplace=True, drop=True) # reset the index to ensure that it matches the index of our dataframe following neighborhood analysis

We need to ensure that the neighborhood analysis dataframe is in a format that allows for rounding that will result in real values when we calculate mean and standard deviation of the annotated neighborhood centroids. Type 'float32' should be sufficient based on previous investigation.

In [ ]:
# List of cell type columns
cell_type_columns = ['CD73+ Stromal', 'Endothelial',
       'Neutrophil', 'DC', 'FAP+ Fibroblast', 'CD163+ CD206+ Macrophage',
       'Stromal', 'CD56+ Tumor', 'Smooth Muscle', 'Epithelial', 'CD8+ T',
       'CD68+ Macrophage', 'CD4+ T', 'CD4+ Treg', 'GZMB+ CD8+ T',
       'TCF1/7+ CD8+ T', 'GZMB+ PD1+ CD8+ T', 'PD1+ CD8+ T',
       'PD1+ TCF1/7+ CD8+ T', 'GZMB+ TCF1/7+ CD8+ T',
       'GZMB+ TCF1/7+ PD1+ CD8+ T', 'NK', 'Nerve', 'Tumor', 'MelanA+ Tumor',
       'SOX10/S100+ Tumor', 'PTEN+ Tumor', 'FAP+ Tumor', 'LDHA+ Tumor',
       'MHCI+ Tumor', 'MHCI+ PDL1+ Tumor', 'EZH2+ Tumor', 'EZH2+ Ki67+ Tumor',
       'PDL1+ CD56+ Tumor', 'Bcatenin+ Tumor', 'PDPN+ Stromal', 'B', 'FDCs',
       'LDHA+ SOX10/S100+ Tumor']
windows2[cell_type_columns] = windows2[cell_type_columns].astype('float32')

Now, we will loop through each neighborhood, filter the annotated dataset for cells that belong to this neighborhood, and calculate the mean and standard deviation for each of the nearest neighbor cell type vectors. This will return a dataframe with neighborhoods as rows and columns with a cell type feature mean and standard deviation for each of the possible cell types in the nearest neighbor vectors.

In [ ]:
# List of neighborhoods to loop through
neighborhoods_to_loop = ['Stromal Enriched', 'Vasculature', 'Neutrophil Enriched',
       'Macrophage Enriched Immune', 'Vasculature & Immune',
       'Productive T cell & Tumor', 'Proliferating Tumor',
       'Inflamed Tumor', 'Tumor & Immune', 'Epithelial/Skin Appendages',
       'Resting Tumor', 'PDPN+ Stromal Enriched', 'Follicle',
       'Perivascular', 'Immune Infiltrate', 'DC Enriched Immune']  # Replace with actual neighborhoods

# List of cell type columns to compute mean and std for
cell_type_columns = ['CD73+ Stromal', 'Endothelial',
       'Neutrophil', 'DC', 'FAP+ Fibroblast', 'CD163+ CD206+ Macrophage',
       'Stromal', 'CD56+ Tumor', 'Smooth Muscle', 'Epithelial', 'CD8+ T',
       'CD68+ Macrophage', 'CD4+ T', 'CD4+ Treg', 'GZMB+ CD8+ T',
       'TCF1/7+ CD8+ T', 'GZMB+ PD1+ CD8+ T', 'PD1+ CD8+ T',
       'PD1+ TCF1/7+ CD8+ T', 'GZMB+ TCF1/7+ CD8+ T',
       'GZMB+ TCF1/7+ PD1+ CD8+ T', 'NK', 'Nerve', 'Tumor', 'MelanA+ Tumor',
       'SOX10/S100+ Tumor', 'PTEN+ Tumor', 'FAP+ Tumor', 'LDHA+ Tumor',
       'MHCI+ Tumor', 'MHCI+ PDL1+ Tumor', 'EZH2+ Tumor', 'EZH2+ Ki67+ Tumor',
       'PDL1+ CD56+ Tumor', 'Bcatenin+ Tumor', 'PDPN+ Stromal', 'B', 'FDCs',
       'LDHA+ SOX10/S100+ Tumor']

# Create an empty list to store the results for each neighborhood
all_results = []

# Loop through each neighborhood
for neighborhood in neighborhoods_to_loop:
    # Step 1: Filter df1 by the specific neighborhood
    filtered_neighborhood_df = filtered_cells[filtered_cells['Neighborhood'] == neighborhood]

    # Step 2: Extract the cell numbers from the filtered rows
    cell_numbers_in_neighborhood = filtered_neighborhood_df.index.values

    # Step 3: Find matching cell numbers in df2
    matching_cells_df = windows2[windows2.index.isin(cell_numbers_in_neighborhood)]

    # Create a dictionary to store the mean and std for the current neighborhood
    mean_std_results = {'Neighborhood': neighborhood}

    # Step 4: Calculate mean and std for each cell type column
    for column in cell_type_columns:
        if column in matching_cells_df.columns:
            column_mean = matching_cells_df[column].mean()
            column_std = matching_cells_df[column].std()
            mean_std_results[f'{column}_mean'] = column_mean
            mean_std_results[f'{column}_std'] = column_std

    # Append the results for this neighborhood to the all_results list
    all_results.append(mean_std_results)

# Step 5: Create a DataFrame from the results list
results_df = pd.DataFrame(all_results)

# Print the resulting DataFrame
print(results_df)

                  Neighborhood  CD73+ Stromal_mean  CD73+ Stromal_std  \
0             Stromal Enriched            0.819380           1.836639   
1                  Vasculature            0.057742           0.283592   
2          Neutrophil Enriched            0.098276           0.374452   
3   Macrophage Enriched Immune            0.287318           0.633737   
4         Vasculature & Immune            0.210323           0.564411   
5    Productive T cell & Tumor            0.042687           0.240209   
6          Proliferating Tumor            0.019477           0.155095   
7               Inflamed Tumor            0.017992           0.155159   
8               Tumor & Immune            0.092882           0.388912   
9   Epithelial/Skin Appendages            0.109522           0.424832   
10               Resting Tumor            0.034558           0.227275   
11      PDPN+ Stromal Enriched            0.095386           0.406414   
12                    Follicle            0.082388 

In [ ]:
# Save centroid results as CSV
results_df.to_csv(r'Z:\MINGLE\Data\melanoma_all_neighborhood_centroids.csv', index=False)